# VERIONCO — NB09 (Unified): Full 6-Rung Ablation — v2

**Changes in v2 (post-diagnostic):**
- Clean C→D ablation: `ORCH_WITH_COUNTER_SYSTEM == RUNG_B_SYSTEM`
- Rung A no longer punts to MDT
- Red-team findings now feed back via a revision pass (rung E)
- Counter-retrieval filter drops neutral clinical-trials hits
- Pathology evidence included in pack (rung B is genuinely multimodal)
- Per-case activation log for risk-adaptive routing metrics
- Token accounting for cost reporting

In [1]:

# Detect environment
IS_KAGGLE = Path("/kaggle").exists()

def find_dir(candidates, fallback):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return Path(fallback)

# ============================================================
# CONFIGURATION
# ============================================================
from pathlib import Path
import os, gc, sys, json, re, time, gc, hashlib
from datetime import datetime, timezone
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

IS_KAGGLE = Path("/kaggle").exists()
OUTPUT_ROOT = Path("/kaggle/working/VERIONCO_outputs") if IS_KAGGLE else Path("./VERIONCO_outputs")
NB_OUTPUT = OUTPUT_ROOT / "09"

PATHS = {
    "nb00": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00").exists()) else OUTPUT_ROOT / "00",
    "nb01": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01").exists()) else OUTPUT_ROOT / "01",
    "nb02": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02").exists()) else OUTPUT_ROOT / "02",
    "nb03": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03").exists()) else OUTPUT_ROOT / "03",
    "nb04": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-04") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-04").exists()) else OUTPUT_ROOT / "04",
    "nb05": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-05") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-05").exists()) else OUTPUT_ROOT / "05",
    "nb06": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-06") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-06").exists()) else OUTPUT_ROOT / "06",
    "nb07": Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-07") if (IS_KAGGLE and Path("/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-07").exists()) else OUTPUT_ROOT / "07",
}

OUTPUT_ROOT = Path("/kaggle/working/VERIONCO_outputs") if Path("/kaggle").exists() else Path("./VERIONCO_outputs")
NB_OUTPUT       = OUTPUT_ROOT / "09"
NB_OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / "09"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

CACHE_DIR = CHECKPOINT_ROOT / "llm_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_LOG   = CHECKPOINT_ROOT / "llm_raw_calls.jsonl"

PROJECT  = "VERIONCO"
NOTEBOOK = "09_unified_ablation"

# ---- LLM settings -------------------------------------------
LLM_MODE          = "api"

# ⚠️  VERIFY THIS with genai.list_models() in the next cell.
#     "gemini-3.5-flash-lite" is NOT a public model id.
#     If your Google AI Studio account exposes a private alias, use it.
#     Otherwise pick from the list printed by the verification cell,
#     e.g. "gemini-2.0-flash", "gemini-1.5-flash-002", etc.
LLM_MODEL         = "gemini-3.5-flash-lite"   # <-- REPLACE WITH REAL SNAPSHOT

TEMPERATURE       = 0.0
MAX_OUTPUT_TOKENS = 4096
PROMPT_VERSION    = "09_v4"   # v4: stricter red-team + counter-aware second opinion
# Multi-Key & Rate-Limiting Configuration (Supports 8 Keys)
MAX_RPM_PER_KEY   = 15             # Free tier = 15 RPM per key; Paid = 1000+
KEY_COOLDOWN_SEC  = 60.0           # Cooldown seconds when 429 quota is hit

API_KEY_NAMES = [
    "GEMINI_API_KEY",
    "GEMINI_API_KEY_2",
    "GEMINI_API_KEY_3",
    "GEMINI_API_KEY_4",
    "GEMINI_API_KEY_5",
    "GEMINI_API_KEY_6",
    "GEMINI_API_KEY_7",
    "GEMINI_API_KEY_8",
]

MANUAL_API_KEYS = [
    # "AIzaSy...",
]

KEY_EXHAUSTION_THRESHOLD = 3
ALL_EXHAUSTED_SLEEP_S    = 60

# ---- Run controls -------------------------------------------
MAX_PATIENTS        = None  # Full dataset: 478 cases (None = no cap, processes all cohort patients)
TIME_BUDGET_S       = 11 * 3600
SLEEP_BETWEEN_CALLS = 2.0

RUNGS_TO_RUN = ["A", "B", "C", "D", "E", "F"]

# ---- Evidence compression -----------------------------------
TOP_GENES          = 20
TOP_SUPPORT_HITS   = 5
TOP_COUNTER_HITS   = 5
TOP_GUIDELINE_HITS = 5
TOP_CONFLICTS      = 10

# ---- Safety gate (rung F) -----------------------------------
REDTEAM_SEVERE_THRESHOLD    = 2
MIN_PRIMARY_CONFIDENCE      = 0.40
MIN_INDEPENDENT_CONFIDENCE  = 0.50

for k, v in PATHS.items():
    print(f"{k:10s} | {v} | exists={v.exists()}")

missing = [k for k, v in PATHS.items() if not v.exists()]
if missing:
    print(f"⚠️  Missing input roots: {missing} — proceeding (some may be optional)")

print(f"\nOutput:      {NB_OUTPUT}")
print(f"Cache:       {CACHE_DIR}")
print(f"LLM:         {LLM_MODEL} | prompt_version={PROMPT_VERSION}")
print(f"Rungs:       {RUNGS_TO_RUN}")
print(f"API keys:    {API_KEY_NAMES}")

nb00       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00 | exists=True
nb01       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-01 | exists=True
nb02       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-02 | exists=True
nb03       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-03 | exists=True
nb04       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-04 | exists=True
nb05       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-05 | exists=True
nb06       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-06 | exists=True
nb07       | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-07 | exists=True

Output:      /kaggle/working/VERIONCO_outputs/09
Cache:       /kaggle/working/VERIONCO_outputs/checkpoints/09/llm_cache
LLM:         gemini-3.5-flash-lite | prompt_version=09_v4
Rungs:       ['A', 'B', 'C', 'D', 'E', 'F']
API keys:    ['GEMINI_API_KEY', 'GEMINI_API_KEY_2']


In [2]:
import os, sys, json
import pandas as pd
from datetime import datetime, timezone

def load_available_keys(names, manual_keys=None):
    found = []
    
    # 0. Try central gemini_api_keys.json file (Single common file for all notebooks)
    key_file_candidates = [
        Path("gemini_api_keys.json"),
        Path("../gemini_api_keys.json"),
        Path("./data/gemini_api_keys.json"),
        Path("/kaggle/working/gemini_api_keys.json"),
        Path("/kaggle/input/gemini_api_keys.json"),
    ]
    for kfc in key_file_candidates:
        if kfc.exists():
            try:
                with open(kfc, "r", encoding="utf-8") as f:
                    content = json.load(f)
                    if isinstance(content, list):
                        for idx, k in enumerate(content, 1):
                            k_str = str(k).strip()
                            if len(k_str) > 10 and not k_str.startswith("YOUR_") and not any(key == k_str for _, key in found):
                                found.append((f"COMMON_FILE_KEY_{idx}", k_str))
                                print(f"✓ Key #{idx} loaded from common file {kfc.name} (length={len(k_str)})")
                        if found:
                            break
            except Exception as e:
                print(f"Warning: could not read {kfc}: {e}")

    # 1. Try Kaggle UserSecretsClient
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for name in names:
            try:
                k = secrets.get_secret(name)
                if k and len(k) > 10 and not any(key == k.strip() for _, key in found):
                    found.append((name, k.strip()))
                    print(f"✓ {name} loaded from Kaggle Secrets (length={len(k)})")
            except Exception:
                pass
    except Exception:
        pass

    # 2. Try OS Environment variables
    for name in names:
        if name in os.environ and not any(n == name for n, _ in found):
            k = os.environ[name].strip()
            if len(k) > 10 and not any(key == k for _, key in found):
                found.append((name, k))
                print(f"✓ {name} loaded from environment (length={len(k)})")

    # 3. Add manual keys if provided
    if manual_keys:
        for idx, k in enumerate(manual_keys, 1):
            k_str = str(k).strip()
            if len(k_str) > 10 and not k_str.startswith("YOUR_") and not any(key == k_str for _, key in found):
                name = f"MANUAL_KEY_{idx}"
                found.append((name, k_str))
                print(f"✓ {name} loaded manually (length={len(k_str)})")

    return found

AVAILABLE_KEYS = load_available_keys(API_KEY_NAMES, MANUAL_API_KEYS) if LLM_MODE == "api" else []

if not AVAILABLE_KEYS:
    print("\n⚠️  No API keys available. Falling back to LLM_MODE='none'.")
    LLM_MODE = "none"
else:
    print(f"\n>>> Loaded {len(AVAILABLE_KEYS)} active API keys!")
    print(f">>> Aggregate throughput capacity: {len(AVAILABLE_KEYS) * MAX_RPM_PER_KEY} RPM ({len(AVAILABLE_KEYS) * MAX_RPM_PER_KEY / 60.0:.2f} calls/sec)")

print(json.dumps({
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "llm_mode": LLM_MODE,
    "n_keys_loaded": len(AVAILABLE_KEYS),
}, indent=2))


✓ GEMINI_API_KEY loaded (length=53)
✓ GEMINI_API_KEY_2 loaded (length=53)


/usr/local/lib/python3.13/dist-packages/wrapt/importer.py:184: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)



Available generateContent models:
  models/gemini-2.5-flash
  models/gemini-2.5-pro
  models/gemini-2.5-flash-preview-tts
  models/gemini-2.5-pro-preview-tts
  models/gemma-4-26b-a4b-it
  models/gemma-4-31b-it
  models/gemini-flash-latest
  models/gemini-flash-lite-latest
  models/gemini-pro-latest
  models/gemini-2.5-flash-lite
  models/gemini-2.5-flash-image
  models/gemini-3-flash-preview
  models/gemini-3.1-pro-preview
  models/gemini-3.1-pro-preview-customtools
  models/gemini-3.1-flash-lite-preview
  models/gemini-3.1-flash-lite
  models/gemini-3-pro-image-preview
  models/gemini-3-pro-image
  models/nano-banana-pro-preview
  models/gemini-3.1-flash-image-preview
  models/gemini-3.1-flash-image
  models/gemini-3.1-flash-lite-image
  models/gemini-nano-banana-2.1
  models/gemini-3.5-flash
  models/gemini-3.5-flash-lite
  models/gemini-omni-flash-preview
  models/gemini-omni-1.1-flash
  models/gemini-3.5-transcribe
  models/gemini-3.6-flash
  models/gemini-3.7-flash
  models/gemin

In [3]:
!pip install -q --upgrade google-generativeai
!pip install -q json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00


In [4]:
# ============================================================
# LOAD — upstream artifacts
# ============================================================
t0 = time.time()

def _read_csv(p, **kw):
    p = Path(p)
    if not p.exists():
        print(f"  MISSING: {p}")
        return pd.DataFrame()
    return pd.read_csv(p, **kw)

cohort = _read_csv(PATHS["nb00"] / "cohort_joined_wsi.csv")
if "filename" in cohort.columns and "file_name" not in cohort.columns:
    cohort = cohort.rename(columns={"filename": "file_name"})
patient_set = set(cohort["patient_id"].astype(str).unique())
N_CASES, N_PATIENTS = len(cohort), len(patient_set)
print(f"[00] cohort: {N_CASES} cases | {N_PATIENTS} patients")

mutations          = _read_csv(PATHS["nb02"] / "patient_mutations.csv", low_memory=False)
civic_var          = _read_csv(PATHS["nb02"] / "civic_matches_VARIANT_LEVEL.csv")
clinical_features  = _read_csv(PATHS["nb03"] / "clinical_features.csv")
conflicts          = _read_csv(PATHS["nb04"] / "conflicts.csv")
retrieval_hits     = _read_csv(PATHS["nb05"] / "retrieval_hits.csv", low_memory=False)
nb06_flat          = _read_csv(PATHS["nb06"] / "plans_flat.csv")

print(f"[02] mutations={mutations.shape} civic_var={civic_var.shape}")
print(f"[03] clinical_features={clinical_features.shape}")
print(f"[04] conflicts={conflicts.shape}")
print(f"[05] retrieval_hits={retrieval_hits.shape}")
print(f"[06] plans_flat={nb06_flat.shape}")

nb06_plans = {}
for p in (PATHS["nb06"] / "plans_per_patient").glob("*.json"):
    with open(p, "r", encoding="utf-8") as f:
        nb06_plans[p.stem] = json.load(f)
print(f"[06] per-patient plans: {len(nb06_plans)}")

print(f"\n[✓] loaded in {time.time()-t0:.1f}s")

[00] cohort: 98 cases | 98 patients
[02] mutations=(32040, 119) civic_var=(80, 14)
[03] clinical_features=(98, 33)
[04] conflicts=(214, 16)
[05] retrieval_hits=(47158, 13)
[06] plans_flat=(98, 13)
[06] per-patient plans: 98

[✓] loaded in 2.5s


In [5]:
# ============================================================
# Helper functions (gene-aware ranker, evidence pack, pathology,
#                  counter-channel filter)
# ============================================================
import ast

NAMED_DRUGS = [
    "pembrolizumab", "nivolumab", "atezolizumab", "durvalumab", "ipilimumab",
    "osimertinib", "erlotinib", "gefitinib", "afatinib", "dacomitinib", "icotinib",
    "crizotinib", "alectinib", "lorlatinib", "ceritinib", "brigatinib",
    "dabrafenib", "trametinib", "vemurafenib",
    "sotorasib", "adagrasib",
    "carboplatin", "cisplatin", "pemetrexed", "paclitaxel",
    "docetaxel", "bevacizumab", "ramucirumab",
    "trastuzumab", "ado-trastuzumab", "entrectinib", "larotrectinib",
    "capmatinib", "tepotinib", "selpercatinib", "pralsetinib",
]

GENE_DRUG_MAP = {
    "ALK":   ["crizotinib", "alectinib", "ceritinib", "lorlatinib", "brigatinib"],
    "ROS1":  ["crizotinib", "entrectinib", "lorlatinib", "ceritinib"],
    "MET":   ["crizotinib", "capmatinib", "tepotinib"],
    "EGFR":  ["osimertinib", "erlotinib", "gefitinib", "afatinib", "dacomitinib", "icotinib"],
    "BRAF":  ["dabrafenib", "trametinib", "vemurafenib"],
    "KRAS":  ["sotorasib", "adagrasib"],
    "RET":   ["selpercatinib", "pralsetinib"],
    "ERBB2": ["trastuzumab", "ado-trastuzumab"],
    "NTRK1": ["entrectinib", "larotrectinib"],
    "NTRK2": ["entrectinib", "larotrectinib"],
    "NTRK3": ["entrectinib", "larotrectinib"],
    "PIK3CA": ["alpelisib"],
}

DRUG_TARGET_GENES = {}
for gene, drugs in GENE_DRUG_MAP.items():
    for d in drugs:
        DRUG_TARGET_GENES.setdefault(d.lower(), set()).add(gene.upper())


def _filter_by_patient(df, pid):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        return pd.DataFrame()
    if "patient_id" not in df.columns:
        return pd.DataFrame()
    return df[df["patient_id"].astype(str) == str(pid)]


def _patient_functional_genes(pid):
    FUNCTIONAL_CLASSES = {
        "Missense_Mutation", "Nonsense_Mutation", "Frame_Shift_Del",
        "Frame_Shift_Ins", "Splice_Site", "In_Frame_Del", "In_Frame_Ins",
        "Nonstop_Mutation", "Translation_Start_Site",
    }
    p_muts = _filter_by_patient(mutations, pid)
    if not len(p_muts):
        return set()
    if "Variant_Classification" in p_muts.columns:
        p_muts = p_muts[p_muts["Variant_Classification"].isin(FUNCTIONAL_CLASSES)]
    if not len(p_muts) or "Hugo_Symbol" not in p_muts.columns:
        return set()
    return {str(g).upper() for g in p_muts["Hugo_Symbol"] if pd.notna(g)}


def get_top_genes(pid, top_n=TOP_GENES):
    p_muts = _filter_by_patient(mutations, pid)
    if not len(p_muts):
        return []
    if "tumor_vaf" in p_muts.columns:
        p_muts = p_muts.sort_values("tumor_vaf", ascending=False, na_position="last")
    seen, out = set(), []
    for _, r in p_muts.iterrows():
        g = str(r.get("Hugo_Symbol", "")).strip()
        if g and g.upper() not in seen:
            seen.add(g.upper())
            out.append(g)
        if len(out) >= top_n:
            break
    return out


def rank_therapy_candidates(pid, evidence_pack):
    candidates = {}
    patient_genes = _patient_functional_genes(pid)

    p_civic = _filter_by_patient(civic_var, pid)
    if len(p_civic) and "civic_therapies" in p_civic.columns:
        for _, r in p_civic.iterrows():
            gene = str(r.get("gene", "")).upper()
            therapies_str = str(r.get("civic_therapies", "") or "")
            for raw_drug in re.split(r"[,;|]", therapies_str):
                drug = raw_drug.strip()
                if not drug or len(drug) < 3:
                    continue
                drug_key = drug.lower()
                target_genes = DRUG_TARGET_GENES.get(drug_key, set())
                if not target_genes or not (target_genes & patient_genes):
                    continue
                if drug_key not in candidates:
                    candidates[drug_key] = {"drug": drug.capitalize(), "score": 0, "sources": []}
                candidates[drug_key]["score"] += 10 if gene in target_genes else 3
                candidates[drug_key]["sources"].append(
                    f"civic:{r.get('civic_id', r.get('id', '?'))}")

    for h in evidence_pack.get("guideline_hits", []):
        title = str(h.get("title", "")).lower()
        for drug_key, target_genes in DRUG_TARGET_GENES.items():
            if drug_key in title and (target_genes & patient_genes):
                if drug_key not in candidates:
                    candidates[drug_key] = {"drug": drug_key.capitalize(), "score": 0, "sources": []}
                candidates[drug_key]["score"] += 5
                candidates[drug_key]["sources"].append(f"guideline:{h['source_id']}")

    clin = evidence_pack.get("clinical", {}) or {}
    tmb = clin.get("tmb")
    try:
        tmb_high = float(tmb) >= 10.0 if tmb is not None else False
    except (TypeError, ValueError):
        tmb_high = False

    if tmb_high or len(candidates) == 0:
        for drug in ["pembrolizumab", "nivolumab", "atezolizumab"]:
            candidates.setdefault(drug, {
                "drug": drug.capitalize(),
                "score": 5 if tmb_high else 3,
                "sources": ["guideline:standard_of_care"],
            })

    if len(candidates) < 3 or all(c["score"] < 5 for c in candidates.values()):
        candidates.setdefault("carboplatin+pemetrexed", {
            "drug": "Carboplatin+Pemetrexed", "score": 4,
            "sources": ["guideline:platinum_doublet"],
        })
        candidates.setdefault("cisplatin+pemetrexed", {
            "drug": "Cisplatin+Pemetrexed", "score": 3,
            "sources": ["guideline:platinum_doublet"],
        })

    ranked = sorted(candidates.values(), key=lambda x: -x["score"])
    return ranked[:5]


def format_candidates(candidates):
    if not candidates:
        return "  - none"
    return "\n".join(
        f"  {i}. {c['drug']} (score {c['score']}; sources: {', '.join(c['sources'][:3])})"
        for i, c in enumerate(candidates, 1)
    )


# ---- Pathology evidence (from NB01) -------------------------
_pathology_cache = {}

def load_pathology_evidence(pid):
    """Return pathology evidence dict for patient, or {} if unavailable."""
    if pid in _pathology_cache:
        return _pathology_cache[pid]
    out = {}
    root = PATHS.get("nb01")
    if root is None or not Path(root).exists():
        _pathology_cache[pid] = out
        return out
    candidates = [
        Path(root) / "per_patient" / f"{pid}.json",
        Path(root) / f"{pid}_pathology.json",
        Path(root) / "pathology_evidence" / f"{pid}.json",
    ]
    for c in candidates:
        if c.exists():
            try:
                with open(c, "r", encoding="utf-8") as f:
                    out = json.load(f)
                break
            except Exception:
                pass
    _pathology_cache[pid] = out
    return out


# ---- Counter-channel filter -------------------------------
NEG_STATUS = {"terminated", "withdrawn", "suspended"}
NEG_KW = (
    "negative", "failed", "no benefit", "did not improve",
    "not superior", "phase iii fail", "resistance", "refractory",
    "contraindicat",
)

def _is_counter_relevant(row) -> bool:
    """Return True if a retrieval hit is genuinely adversarial.
    Non-clinicaltrials sources pass through (filtered upstream)."""
    src = str(row.get("source", "")).lower()
    if src == "clinicaltrials":
        status = str(row.get("study_status", "")).lower()
        if status in NEG_STATUS:
            return True
        title = str(row.get("title", "")).lower()
        return any(kw in title for kw in NEG_KW)
    return True


def load_evidence_pack(pid):
    p_hits = _filter_by_patient(retrieval_hits, pid)
    p_clin = _filter_by_patient(clinical_features, pid)
    p_conf = _filter_by_patient(conflicts, pid)

    clin = p_clin.iloc[0].to_dict() if len(p_clin) else {}

    conf_list = []
    if len(p_conf):
        p_conf = p_conf[p_conf["severity"].isin(["high", "medium"])]
        p_conf = p_conf.head(TOP_CONFLICTS)
        conf_list = p_conf[["conflict_type", "severity", "description"]].to_dict("records")

    support_hits_all = p_hits[p_hits["retrieval_channel"] == "support"] if len(p_hits) else pd.DataFrame()
    counter_hits_all = p_hits[p_hits["retrieval_channel"] == "counter"] if len(p_hits) else pd.DataFrame()
    guideline_hits = pd.DataFrame()
    if len(p_hits) and "source" in p_hits.columns:
        guideline_hits = p_hits[p_hits["source"] == "guideline"]

    # Filter counter channel to genuinely adversarial hits only
    if len(counter_hits_all):
        keep_mask = counter_hits_all.apply(_is_counter_relevant, axis=1)
        counter_hits_all = counter_hits_all[keep_mask]

    path_ev = load_pathology_evidence(pid)

    return {
        "patient_id": pid,
        "clinical": clin,
        "conflicts": conf_list,
        "top_genes": get_top_genes(pid),
        "pathology": path_ev,
        "support_hits": support_hits_all.head(TOP_SUPPORT_HITS).to_dict("records") if len(support_hits_all) else [],
        "counter_hits": counter_hits_all.head(TOP_COUNTER_HITS).to_dict("records") if len(counter_hits_all) else [],
        "guideline_hits": guideline_hits.head(TOP_GUIDELINE_HITS).to_dict("records") if len(guideline_hits) else [],
    }


print("✓ helper functions defined")

✓ helper functions defined


In [6]:
# ============================================================
# LLM CLIENT — 8-Key Load-Balanced & Per-Minute Rate-Limited
# ============================================================
import subprocess, time, hashlib, json, re
from collections import deque
subprocess.run(["pip", "install", "-q", "json-repair"], check=True)
from json_repair import repair_json

def _hash(*parts):
    h = hashlib.sha256()
    for p in parts:
        h.update(str(p).encode("utf-8"))
    return h.hexdigest()[:16]

def hash_evidence(pack):
    canonical = json.dumps(pack, sort_keys=True, default=str)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]

class LLMError(Exception):
    pass

class RateLimitedMultiKeyGeminiClient:
    def __init__(self, keys, model=LLM_MODEL, max_rpm_per_key=MAX_RPM_PER_KEY, max_retries=12):
        import google.generativeai as genai
        self._genai = genai
        self.model_name = model
        self.max_retries = max_retries
        self.max_rpm = max_rpm_per_key

        if not keys:
            raise LLMError("No API keys provided")

        self.keys = list(keys)
        self.n_keys = len(self.keys)

        # Sliding window timestamp queues for each key (last 60s)
        self.key_history = {name: deque() for name, _ in self.keys}
        self.key_cooldown_until = {name: 0.0 for name, _ in self.keys}
        self.key_calls = {name: 0 for name, _ in self.keys}
        self.key_errors = {name: 0 for name, _ in self.keys}
        self.key_429s = {name: 0 for name, _ in self.keys}

        self.calls_this_run = 0
        self.cache_hits = 0
        self.json_repairs = 0
        self.rotations = 0
        self.exhaustion_events = 0
        self.exhausted = {name: False for name, _ in self.keys}

        self.tokens_in = 0
        self.tokens_out = 0

        self.current_idx = 0
        self._configure_key(0)

    def _configure_key(self, idx):
        name, key = self.keys[idx]
        self._genai.configure(api_key=key)
        self._current_key_name = name

    def _clean_history(self, name, now):
        q = self.key_history[name]
        cutoff = now - 60.0
        while q and q[0] < cutoff:
            q.popleft()

    def _find_available_key(self):
        """Find a key that is not in cooldown and has < max_rpm in the last 60s."""
        while True:
            now = time.time()
            best_key_idx = None
            min_calls = 999999

            for offset in range(self.n_keys):
                idx = (self.current_idx + offset) % self.n_keys
                name, _ = self.keys[idx]

                if now < self.key_cooldown_until[name]:
                    continue

                self._clean_history(name, now)
                current_rpm = len(self.key_history[name])

                if current_rpm < self.max_rpm:
                    if current_rpm < min_calls:
                        min_calls = current_rpm
                        best_key_idx = idx
                        if current_rpm == 0:
                            break

            if best_key_idx is not None:
                if best_key_idx != self.current_idx:
                    self.current_idx = best_key_idx
                    self.rotations += 1
                    self._configure_key(self.current_idx)
                return self.keys[self.current_idx][0]

            # All keys full or on cooldown -> compute wait time
            earliest_free = 999999.0
            for name, _ in self.keys:
                if now < self.key_cooldown_until[name]:
                    wait_cd = self.key_cooldown_until[name] - now
                    if wait_cd < earliest_free:
                        earliest_free = wait_cd
                else:
                    self._clean_history(name, now)
                    q = self.key_history[name]
                    if q:
                        wait_rpm = 60.0 - (now - q[0]) + 0.1
                        if wait_rpm < earliest_free:
                            earliest_free = wait_rpm

            wait_time = max(0.5, min(earliest_free, 10.0))
            time.sleep(wait_time)

    def _record_429(self, key_name):
        now = time.time()
        self.key_429s[key_name] += 1
        self.exhaustion_events += 1
        self.exhausted[key_name] = True
        self.key_cooldown_until[key_name] = now + KEY_COOLDOWN_SEC
        print(f"    ⛔ Key '{key_name}' hit rate limit (429). Cooldown for {KEY_COOLDOWN_SEC:.0f}s. Rotating...")

    def _record_success(self, key_name):
        now = time.time()
        self.key_history[key_name].append(now)
        self.key_calls[key_name] += 1
        self.calls_this_run += 1

    def _cache_path(self, key):
        return CACHE_DIR / f"{key}.json"

    @staticmethod
    def _repair_with_fallback(raw):
        try:
            fixed = repair_json(raw, return_objects=False)
            json.loads(fixed)
            return fixed
        except Exception:
            pass
        s = raw.rstrip()
        try:
            json.loads(s)
            return s
        except json.JSONDecodeError:
            pass
        if s.count('"') % 2 == 1:
            s += '"'
        for opener, closer in [("{", "}"), ("[", "]")]:
            opens = s.count(opener) - s.count(closer)
            s += closer * max(0, opens)
        return s

    def generate_json(self, prompt, system_prompt, cache_key_parts):
        cache_key = _hash(*cache_key_parts)
        cp = self._cache_path(cache_key)
        if cp.exists():
            with open(cp, "r", encoding="utf-8") as f:
                self.cache_hits += 1
                return json.load(f)

        last_err = None
        attempt = 0
        while attempt < self.max_retries:
            key_name = self._find_available_key()
            try:
                model = self._genai.GenerativeModel(
                    self.model_name,
                    system_instruction=system_prompt,
                    generation_config={
                        "temperature": TEMPERATURE,
                        "max_output_tokens": MAX_OUTPUT_TOKENS,
                        "response_mime_type": "application/json",
                    },
                )
                resp = model.generate_content(prompt)
                raw = resp.text
                self._record_success(key_name)

                # token accounting
                usage = getattr(resp, "usage_metadata", None)
                if usage is not None:
                    self.tokens_in  += int(getattr(usage, "prompt_token_count", 0) or 0)
                    self.tokens_out += int(getattr(usage, "candidates_token_count", 0) or 0)

                raw_clean = re.sub(r"^```json\s*", "", raw.strip())
                raw_clean = re.sub(r"\s*```$", "", raw_clean)

                try:
                    data = json.loads(raw_clean)
                except json.JSONDecodeError:
                    repaired = self._repair_with_fallback(raw_clean)
                    data = json.loads(repaired)
                    self.json_repairs += 1

                # Save cache
                with open(cp, "w", encoding="utf-8") as f:
                    json.dump(data, f, indent=2)

                # Log call
                with open(RAW_LOG, "a", encoding="utf-8") as f:
                    f.write(json.dumps({
                        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                        "cache_key": cache_key,
                        "key_used": key_name,
                        "raw_response": raw,
                    }) + "\n")

                return data

            except Exception as e:
                err_str = str(e).lower()
                if "429" in err_str or "quota" in err_str or "resource_exhausted" in err_str:
                    self._record_429(key_name)
                else:
                    self.key_errors[key_name] += 1
                    print(f"    ⚠️ Warning on {key_name}: {e}. Retrying with next key...")
                last_err = e
                attempt += 1
                time.sleep(1.0)

        raise LLMError(f"Failed after {self.max_retries} attempts across keys. Last error: {last_err}")


llm = RateLimitedMultiKeyGeminiClient(AVAILABLE_KEYS) if LLM_MODE == "api" and AVAILABLE_KEYS else None


✓ LLM client ready: True
  keys: ['GEMINI_API_KEY', 'GEMINI_API_KEY_2']


In [7]:
# ============================================================
# PROMPTS — rung A, B, C/D, E, E-revision  (v3)
# ============================================================

# ---- Rung A: no retrieval, no punt ------------------------
RUNG_A_SYSTEM = """You are a research assistant for a decision-support system.
Output ONLY valid JSON. Do not invent papers, trials, guidelines, or clinical facts.
You have only clinical and genomic facts. No retrieval.

You MUST name a specific recommended therapy. Do NOT use ESCALATE_TO_MDT.
If evidence is thin, pick the most defensible standard-of-care option for
LUAD (platinum doublet, PD-1 inhibitor, or targeted therapy matching an
identified driver) and lower your confidence accordingly. Set
escalation_flag=true only if the case cannot be handled with any LUAD SOC
option."""

# ---- Rung B: orchestrator with support + guideline --------
RUNG_B_SYSTEM = """You are the orchestrator of a decision-support system.
Output ONLY valid JSON. Cite only source_ids explicitly provided.

STRICT RULES:
1. You MUST select from CANDIDATE_THERAPIES if non-empty. Copy the name verbatim.
2. If empty, use "ESCALATE_TO_MDT".
3. Do NOT use "perform more testing".
4. rationale must cite at least one source_id from the input."""

# ---- Rung D: IDENTICAL system prompt as B (clean ablation) ----
# The ONLY difference between C and D is the presence of COUNTER_HITS
# and the operational instruction in the user prompt.
ORCH_WITH_COUNTER_SYSTEM = RUNG_B_SYSTEM

# ---- Second opinion ---------------------------------------
SECOND_OPINION_SYSTEM = """You are an INDEPENDENT second-opinion agent.
Output ONLY valid JSON. Cite only source_ids explicitly provided.
You have NOT seen any other agent's recommendation. Reason independently.

STRICT RULES:
1. If CIVIC_VARIANT_MATCHES supports a targeted therapy, name it.
2. Else, select a STANDARD_OF_CARE_OPTION by name.
3. Reserve "ESCALATE_TO_MDT" for genuinely contradictory or absent evidence.
4. Do NOT use "perform more testing"."""

# v4: counter-aware variant for rung D (only used when counter block is present)
SECOND_OPINION_SYSTEM_WITH_COUNTER = SECOND_OPINION_SYSTEM + """

ADDITIONAL RULE — COUNTER_EVIDENCE is present in the input:
If any COUNTER_EVIDENCE hit directly contradicts one of the
STANDARD_OF_CARE_OPTIONS, prefer a different option that the counter-
evidence does not contradict, or set escalation_flag=true.
Cite the contradicting source_id(s) in "counter_evidence_considered".
If no counter-hit applies to any candidate, set
"counter_evidence_considered" to [] and proceed normally."""

# ---- Red-team adversary (v4: stricter survival criteria) -----
REDTEAM_SYSTEM = """You are a red-team adversary. Your job is to DISPROVE the
primary recommendation. Output ONLY valid JSON. Cite only source_ids provided.

For each attack:
- attack_type: contraindication | resistance | missing_biomarker | weak_evidence |
  staging_issue | trial_exclusion | missing_patient_info | other
- severity: low | medium | high | critical
- evidence_cited: list of source_ids
- explanation: 1-2 sentences
- survives: set to FALSE only if the primary recommendation is clinically
  defensible despite the attack (i.e. the attack is a caveat, not a
  contradiction). Set to TRUE if the primary recommendation, AS WRITTEN,
  could be clinically wrong or harmful given the cited evidence.

IMPORTANT CALIBRATION:
- A red-team that never lands an attack is not doing its job.
- If you find that the primary recommendation lacks direct evidence for the
  specific patient's molecular profile, that is a surviving weak_evidence
  attack (survives=true) — even if the recommendation is generically
  reasonable for LUAD.
- If you find a contraindication or resistance mechanism that applies to
  the specific patient, that is a surviving attack (survives=true).
- Aim for roughly one third of high-severity attacks to survive. Attacks
  that merely note generic guideline reliance (without a specific
  contradiction) should be marked survives=false.

Then redteam_pass: true if NO attack with severity >= high survives."""

# ---- Revision pass (v5: constrained to candidate list) -----
REVISION_SYSTEM = """You are revising a treatment recommendation after an
adversarial red-team review. Output ONLY valid JSON. Cite only source_ids
explicitly provided.

CRITICAL CONSTRAINT:
If you CHANGE the recommendation, the new recommended_treatment MUST be
one of the CANDIDATE_THERAPIES provided below. Do NOT invent new drug
names, do NOT combine therapies in new ways, do NOT use brand-name
variants. Copy the candidate name verbatim.

If none of the CANDIDATE_THERAPIES is defensible given the surviving
attacks, set recommended_treatment = "ESCALATE_TO_MDT".

You are given the original recommendation and the red-team findings.

If you KEEP the original recommendation despite surviving high/critical
attacks, you MUST include a field "defense" (>= 80 words) explaining why
each surviving attack does not materially change the risk/benefit. If you
cannot produce a credible defense, change to the next-best candidate or
escalate.

If you CHANGE the recommendation, cite the surviving attack that drove
the change in "revision_reason"."""


# ============================================================
# Prompt builders
# ============================================================

def _clinical_line(clin):
    return (f"age={clin.get('age')}, sex={clin.get('sex')}, "
            f"stage={clin.get('stage_str')}, "
            f"tmb={clin.get('tmb')}, msi={clin.get('msi_score')}")


def _fmt_hits(hits, label):
    if not hits:
        return f"{label}: none"
    return f"{label}: " + "; ".join(
        f"{h['source']}:{h['source_id']} \"{str(h.get('title',''))[:60]}\""
        for h in hits)


def _pathology_block(pack):
    path = pack.get("pathology", {})
    if not path:
        return "PATHOLOGY_EVIDENCE: unavailable"
    return "PATHOLOGY_EVIDENCE: " + json.dumps(path, default=str)[:800]


def build_rung_a_prompt(pid, evidence_pack):
    genes = ", ".join(evidence_pack["top_genes"]) or "none"
    return f"""PATIENT: {pid}
CANCER: LUAD
CLINICAL: {_clinical_line(evidence_pack['clinical'])}
TOP_MUTATED_GENES: {genes}

Recommend ONE specific named therapy. Do NOT punt to "further testing".
Do NOT use ESCALATE_TO_MDT unless no LUAD SOC option applies.

Return JSON:
{{"recommendation":{{"recommended_treatment":"...","rationale":"...","confidence":0.0,"escalation_flag":false}}}}"""


def build_rung_b_prompt(pid, evidence_pack, nb06_plan):
    candidates = rank_therapy_candidates(pid, evidence_pack)
    candidates_txt = format_candidates(candidates)

    if nb06_plan is None:
        hyps_txt, guide_txt, plans_txt = "none", "none", "none"
    else:
        hyps_txt = json.dumps(nb06_plan.get("hypotheses", {}).get("hypotheses", []),
                              default=str)[:1000]
        guide_txt = json.dumps(nb06_plan.get("guideline_plan", {}).get("items", []),
                               default=str)[:700]
        plans_txt = json.dumps(nb06_plan.get("evidence_plan", {}).get("items", [])[:6],
                               default=str)[:1200]

    return f"""PATIENT: {pid}
CLINICAL: {_clinical_line(evidence_pack['clinical'])}
CONFLICTS: {'; '.join(c['conflict_type'] + '(' + c['severity'] + ')' for c in evidence_pack['conflicts']) or 'none'}
{_pathology_block(evidence_pack)}

CANDIDATE_THERAPIES (choose from this list if non-empty):
{candidates_txt}

NB06 HYPOTHESES:
{hyps_txt}

NB06 GUIDELINE ITEMS:
{guide_txt}

NB06 PLAN ITEMS:
{plans_txt}

{_fmt_hits(evidence_pack['support_hits'], 'SUPPORT_HITS')}
{_fmt_hits(evidence_pack['guideline_hits'], 'GUIDELINE_HITS')}

Return JSON:
{{"recommendation":{{"recommended_treatment":"...","rationale":"...","confidence":0.0,"supporting_citations":["pubmed:..."],"escalation_flag":false}}}}"""


def build_orchestrator_counter_prompt(pid, evidence_pack, nb06_plan):
    """Same structure as rung B + COUNTER_HITS block + operational instruction.
    System prompt identical to rung B (clean ablation)."""
    candidates = rank_therapy_candidates(pid, evidence_pack)
    candidates_txt = format_candidates(candidates)

    if nb06_plan is None:
        hyps_txt, guide_txt, plans_txt = "none", "none", "none"
    else:
        hyps_txt = json.dumps(nb06_plan.get("hypotheses", {}).get("hypotheses", []),
                              default=str)[:1000]
        guide_txt = json.dumps(nb06_plan.get("guideline_plan", {}).get("items", []),
                               default=str)[:700]
        plans_txt = json.dumps(nb06_plan.get("evidence_plan", {}).get("items", [])[:6],
                               default=str)[:1200]

    return f"""PATIENT: {pid}
CLINICAL: {_clinical_line(evidence_pack['clinical'])}
CONFLICTS: {'; '.join(c['conflict_type'] + '(' + c['severity'] + ')' for c in evidence_pack['conflicts']) or 'none'}
{_pathology_block(evidence_pack)}

CANDIDATE_THERAPIES (choose from this list if non-empty):
{candidates_txt}

NB06 HYPOTHESES:
{hyps_txt}

NB06 GUIDELINE ITEMS:
{guide_txt}

NB06 PLAN ITEMS:
{plans_txt}

{_fmt_hits(evidence_pack['support_hits'], 'SUPPORT_HITS')}
{_fmt_hits(evidence_pack['guideline_hits'], 'GUIDELINE_HITS')}
{_fmt_hits(evidence_pack['counter_hits'], 'COUNTER_HITS')}

BEFORE FINALIZING:
Review the COUNTER_HITS block. If any hit directly contradicts your top
CANDIDATE_THERAPY, either:
  (a) select the next-best candidate that the counter-evidence does not
      contradict, or
  (b) set escalation_flag=true and set rationale_cites_counter=true.
If no counter-hit applies to your chosen therapy, proceed normally and set
rationale_cites_counter=false.

Return JSON:
{{"recommendation":{{"recommended_treatment":"...","rationale":"...","confidence":0.0,"supporting_citations":["pubmed:..."],"counter_evidence_acknowledged":["pubmed:..."],"rationale_cites_counter":false,"escalation_flag":false}}}}"""


def build_second_opinion_prompt(pid, evidence_pack, nb06_plan, include_counter):
    p_civic = _filter_by_patient(civic_var, pid)
    civic_txt = "none"
    if len(p_civic) and "civic_therapies" in p_civic.columns:
        civic_txt = "\n".join(
            f"gene={r.get('gene','?')} variant={r.get('variant_canonical','?')} "
            f"therapies={str(r.get('civic_therapies',''))[:100]}"
            for _, r in p_civic.head(10).iterrows())

    clin = evidence_pack.get("clinical", {}) or {}
    try:
        tmb_high = float(clin.get("tmb")) >= 10.0 if clin.get("tmb") is not None else False
    except (TypeError, ValueError):
        tmb_high = False

    soc_txt = "\n".join([
        "  - Pembrolizumab (TMB-high, guideline-supported)" if tmb_high
        else "  - Pembrolizumab (if PD-L1 high)",
        "  - Carboplatin+Pemetrexed (platinum doublet)",
        "  - Cisplatin+Pemetrexed (platinum doublet)",
    ])

    counter_block = ""
    if include_counter:
        counter_block = "\n" + _fmt_hits(
            evidence_pack['counter_hits'],
            'COUNTER_EVIDENCE (weigh against candidates)')

    return f"""PATIENT: {pid}
CLINICAL: {_clinical_line(clin)}
CONFLICTS: {'; '.join(c['conflict_type'] + '(' + c['severity'] + ')' for c in evidence_pack['conflicts']) or 'none'}
{_pathology_block(evidence_pack)}

CIVIC_VARIANT_MATCHES (raw):
{civic_txt}

STANDARD_OF_CARE_OPTIONS:
{soc_txt}

{_fmt_hits(evidence_pack['support_hits'], 'SUPPORT_HITS')}
{_fmt_hits(evidence_pack['guideline_hits'], 'GUIDELINE_HITS')}{counter_block}

Return JSON:
{{"recommendation":{{"recommended_treatment":"...","rationale":"...","confidence":0.0,"supporting_citations":["pubmed:..."],"counter_evidence_considered":[],"escalation_flag":false}}}}"""


def build_redteam_prompt(pid, primary_rec, evidence_pack):
    counter_cits = [f"{h['source']}:{h['source_id']}" for h in evidence_pack['counter_hits']]
    conf_line = "; ".join(
        f"{c['conflict_type']}({c['severity']})" for c in evidence_pack['conflicts']
    ) or "none"

    return f"""PATIENT: {pid}
CLINICAL: {_clinical_line(evidence_pack['clinical'])}

PRIMARY RECOMMENDATION:
{json.dumps(primary_rec, default=str)[:1000]}

COUNTER-EVIDENCE:
{', '.join(counter_cits) if counter_cits else 'none'}

CONFLICTS:
{conf_line}

Return JSON:
{{"redteam_findings":[{{"attack_type":"...","severity":"low|medium|high|critical","evidence_cited":["pubmed:..."],"explanation":"...","survives":true}}],"redteam_pass":false,"redteam_summary":"..."}}"""


def build_revision_prompt(pid, primary_rec, redteam_findings, evidence_pack):
    surviving = [f for f in redteam_findings
                 if f.get("survives") and f.get("severity") in ("high", "critical")]
    candidates = rank_therapy_candidates(pid, evidence_pack)
    candidates_txt = format_candidates(candidates)

    return f"""PATIENT: {pid}
CLINICAL: {_clinical_line(evidence_pack['clinical'])}

CANDIDATE_THERAPIES (if you change, you MUST pick from this list):
{candidates_txt}

ORIGINAL RECOMMENDATION:
{json.dumps(primary_rec, default=str)[:800]}

SURVIVING HIGH/CRITICAL ATTACKS:
{json.dumps(surviving, default=str)[:1500] if surviving else "none"}

{_fmt_hits(evidence_pack['counter_hits'], 'COUNTER_HITS')}

If you KEEP the original recommendation despite surviving high/critical attacks,
you MUST include "defense" (>= 80 words). If you cannot defend it credibly,
change to a candidate from CANDIDATE_THERAPIES or set escalation_flag=true.

Return JSON:
{{"recommendation":{{"recommended_treatment":"...","rationale":"...","confidence":0.0,"escalation_flag":false,"revision_reason":"...","defense":"..."}},"revisions_made":true}}"""

print("✓ prompts defined (v3)")

✓ prompts defined (v3)


In [8]:
# ============================================================
# PIPELINE — one function per rung  (v3)
# ============================================================

def _normalize_recommendation(rec):
    if not isinstance(rec, dict):
        return {"recommended_treatment": "", "rationale": "",
                "confidence": 0.0, "supporting_citations": [],
                "counter_evidence_acknowledged": [], "escalation_flag": False,
                "rationale_cites_counter": False,
                "counter_evidence_considered": [],
                "revision_reason": "", "defense": ""}
    out = dict(rec)
    for k, default in [("recommended_treatment", ""), ("rationale", ""),
                       ("confidence", 0.0), ("escalation_flag", False),
                       ("rationale_cites_counter", False),
                       ("revision_reason", ""), ("defense", "")]:
        out.setdefault(k, default)
    for k in ("supporting_citations", "counter_evidence_acknowledged",
              "counter_evidence_considered"):
        out.setdefault(k, [])
        if not isinstance(out[k], list):
            out[k] = []
    return out


def _norm_treatment_name(s):
    s = str(s).lower().strip()
    s = re.sub(r"\(.*?\)", "", s)
    return re.sub(r"\s+", " ", s).strip()


def run_rung_A(pid, evidence_pack, nb06_plan):
    ev_hash = hash_evidence({"clinical": evidence_pack["clinical"],
                             "top_genes": evidence_pack["top_genes"]})
    result = llm.generate_json(
        prompt=build_rung_a_prompt(pid, evidence_pack),
        system_prompt=RUNG_A_SYSTEM,
        cache_key_parts=[pid, "rung_A", PROMPT_VERSION, LLM_MODEL, ev_hash])
    rec = result.get("recommendation", result) if isinstance(result, dict) else {}
    return {"recommendation": _normalize_recommendation(rec),
            "llm_status": "api_success"}


def run_rung_B(pid, evidence_pack, nb06_plan):
    ev_hash = hash_evidence({"clinical": evidence_pack["clinical"],
                             "top_genes": evidence_pack["top_genes"],
                             "pathology": evidence_pack.get("pathology", {}),
                             "support_hits": evidence_pack["support_hits"],
                             "guideline_hits": evidence_pack["guideline_hits"],
                             "candidates": [c["drug"] for c in rank_therapy_candidates(pid, evidence_pack)]})
    result = llm.generate_json(
        prompt=build_rung_b_prompt(pid, evidence_pack, nb06_plan),
        system_prompt=RUNG_B_SYSTEM,
        cache_key_parts=[pid, "rung_B", PROMPT_VERSION, LLM_MODEL, ev_hash])
    rec = result.get("recommendation", result) if isinstance(result, dict) else {}
    return {"recommendation": _normalize_recommendation(rec),
            "llm_status": "api_success"}


def run_rung_C(pid, evidence_pack, nb06_plan, rung_b_result):
    orch_rec = rung_b_result["recommendation"]

    ev_hash_so = hash_evidence({"clinical": evidence_pack["clinical"],
                                "pathology": evidence_pack.get("pathology", {}),
                                "support_hits": evidence_pack["support_hits"],
                                "guideline_hits": evidence_pack["guideline_hits"]})
    so = llm.generate_json(
        prompt=build_second_opinion_prompt(pid, evidence_pack, nb06_plan, include_counter=False),
        system_prompt=SECOND_OPINION_SYSTEM,
        cache_key_parts=[pid, "rung_C_second_opinion", PROMPT_VERSION, LLM_MODEL, ev_hash_so])
    so_rec = so.get("recommendation", so) if isinstance(so, dict) else {}
    so_rec = _normalize_recommendation(so_rec)

    agreement = (_norm_treatment_name(orch_rec.get("recommended_treatment", "")) ==
                 _norm_treatment_name(so_rec.get("recommended_treatment", "")))

    return {"orchestrator": orch_rec, "second_opinion": so_rec,
            "orch_second_agree": agreement, "llm_status": "api_success"}


def run_rung_D(pid, evidence_pack, nb06_plan):
    ev_hash_orch = hash_evidence({"clinical": evidence_pack["clinical"],
                                  "pathology": evidence_pack.get("pathology", {}),
                                  "support_hits": evidence_pack["support_hits"],
                                  "guideline_hits": evidence_pack["guideline_hits"],
                                  "counter_hits": evidence_pack["counter_hits"]})
    orch = llm.generate_json(
        prompt=build_orchestrator_counter_prompt(pid, evidence_pack, nb06_plan),
        system_prompt=ORCH_WITH_COUNTER_SYSTEM,
        cache_key_parts=[pid, "rung_D_orchestrator", PROMPT_VERSION, LLM_MODEL, ev_hash_orch])
    orch_rec = _normalize_recommendation(
        orch.get("recommendation", orch) if isinstance(orch, dict) else {})

    ev_hash_so = hash_evidence({"clinical": evidence_pack["clinical"],
                                "pathology": evidence_pack.get("pathology", {}),
                                "support_hits": evidence_pack["support_hits"],
                                "guideline_hits": evidence_pack["guideline_hits"],
                                "counter_hits": evidence_pack["counter_hits"]})
    so = llm.generate_json(
        prompt=build_second_opinion_prompt(pid, evidence_pack, nb06_plan, include_counter=True),
        system_prompt=SECOND_OPINION_SYSTEM_WITH_COUNTER,   # v4
        cache_key_parts=[pid, "rung_D_second_opinion_v4", PROMPT_VERSION, LLM_MODEL, ev_hash_so])
    so_rec = _normalize_recommendation(
        so.get("recommendation", so) if isinstance(so, dict) else {})

    agreement = (_norm_treatment_name(orch_rec.get("recommended_treatment", "")) ==
                 _norm_treatment_name(so_rec.get("recommended_treatment", "")))

    return {"orchestrator": orch_rec, "second_opinion": so_rec,
            "orch_second_agree": agreement, "llm_status": "api_success"}


def run_rung_E(pid, evidence_pack, nb06_plan, rung_d_result):
    """Rung E: rung D's outputs + red-team + revision pass (v3: capture defense)."""
    primary_rec = rung_d_result["orchestrator"]

    ev_hash = hash_evidence({"primary_rec": primary_rec,
                             "counter_hits": evidence_pack["counter_hits"],
                             "conflicts": evidence_pack["conflicts"]})
    rt = llm.generate_json(
        prompt=build_redteam_prompt(pid, primary_rec, evidence_pack),
        system_prompt=REDTEAM_SYSTEM,
        cache_key_parts=[pid, "rung_E_redteam", PROMPT_VERSION, LLM_MODEL, ev_hash])

    findings = rt.get("redteam_findings", []) if isinstance(rt, dict) else []
    redteam_pass = bool(rt.get("redteam_pass", len(findings) == 0)) if isinstance(rt, dict) else True

    surviving = [f for f in findings
                 if f.get("survives") and f.get("severity") in ("high", "critical")]
    revisions_made = False
    revised_rec = primary_rec
    defense_text = ""
    revision_reason = ""
    revision_attempted = False

    if surviving:
        revision_attempted = True
        rev_hash = hash_evidence({"primary": primary_rec, "surviving": surviving})
        rev = llm.generate_json(
            prompt=build_revision_prompt(pid, primary_rec, findings, evidence_pack),
            system_prompt=REVISION_SYSTEM,
            cache_key_parts=[pid, "rung_E_revision", PROMPT_VERSION, LLM_MODEL, rev_hash])
        if isinstance(rev, dict):
            revised_rec = _normalize_recommendation(rev.get("recommendation", primary_rec))
            revisions_made = bool(rev.get("revisions_made", False))
            defense_text = str(revised_rec.get("defense", "") or "")
            revision_reason = str(revised_rec.get("revision_reason", "") or "")

    return {"orchestrator": revised_rec,
            "original_orchestrator": primary_rec,
            "second_opinion": rung_d_result["second_opinion"],
            "orch_second_agree": rung_d_result.get("orch_second_agree", False),
            "redteam_findings": findings,
            "redteam_pass": redteam_pass,
            "redteam_summary": rt.get("redteam_summary", "") if isinstance(rt, dict) else "",
            "revision_attempted": revision_attempted,
            "revisions_made": revisions_made,
            "n_surviving_high_critical": len(surviving),
            "defense": defense_text,
            "revision_reason": revision_reason,
            "llm_status": "api_success"}


def run_rung_F(pid, evidence_pack, rung_e_result):
    """Rung F: rung E + deterministic safety gate. 0 API calls."""
    prim = rung_e_result["orchestrator"]
    sec = rung_e_result["second_opinion"]
    findings = rung_e_result.get("redteam_findings", [])
    redteam_pass = rung_e_result.get("redteam_pass", True)

    reasons = []
    if float(prim.get("confidence", 0.0) or 0.0) < MIN_PRIMARY_CONFIDENCE:
        reasons.append("primary_confidence_below_threshold")
    if float(sec.get("confidence", 0.0) or 0.0) < MIN_INDEPENDENT_CONFIDENCE:
        reasons.append("independent_confidence_below_threshold")
    if not rung_e_result.get("orch_second_agree", False):
        reasons.append("primary_independent_disagree_on_drug")

    n_high_crit = sum(1 for f in findings if f.get("severity") in ("high", "critical"))
    if (not redteam_pass) and n_high_crit >= REDTEAM_SEVERE_THRESHOLD:
        reasons.append("redteam_severely_failed")

    n_conf_high = sum(1 for c in evidence_pack["conflicts"] if c.get("severity") == "high")
    if n_conf_high > 2:
        reasons.append(f"high_severity_conflicts_{n_conf_high}")

    decision = "ACCEPT" if len(reasons) == 0 else "ESCALATE"

    return {"orchestrator": prim,
            "second_opinion": sec,
            "orch_second_agree": rung_e_result.get("orch_second_agree", False),
            "redteam_findings": findings,
            "redteam_pass": redteam_pass,
            "safety_gate_decision": decision,
            "safety_gate_reasons": reasons,
            "llm_status": "api_success"}


def process_patient_all_rungs(pid, existing=None):
    evidence_pack = load_evidence_pack(pid)
    nb06_plan = nb06_plans.get(pid)
    out = dict(existing or {})
    out["patient_id"] = pid

    if "A" in RUNGS_TO_RUN and "A" not in out:
        try:
            out["A"] = run_rung_A(pid, evidence_pack, nb06_plan)
        except LLMError as e:
            out["A"] = {"recommendation": _normalize_recommendation({}),
                        "llm_status": "failed_fallback", "llm_error": str(e)}

    if "B" in RUNGS_TO_RUN and "B" not in out:
        try:
            out["B"] = run_rung_B(pid, evidence_pack, nb06_plan)
        except LLMError as e:
            out["B"] = {"recommendation": _normalize_recommendation({}),
                        "llm_status": "failed_fallback", "llm_error": str(e)}

    if "C" in RUNGS_TO_RUN and "C" not in out and "B" in out:
        try:
            out["C"] = run_rung_C(pid, evidence_pack, nb06_plan, out["B"])
        except LLMError as e:
            out["C"] = {"orchestrator": _normalize_recommendation({}),
                        "second_opinion": _normalize_recommendation({}),
                        "orch_second_agree": False,
                        "llm_status": "failed_fallback", "llm_error": str(e)}

    if "D" in RUNGS_TO_RUN and "D" not in out:
        try:
            out["D"] = run_rung_D(pid, evidence_pack, nb06_plan)
        except LLMError as e:
            out["D"] = {"orchestrator": _normalize_recommendation({}),
                        "second_opinion": _normalize_recommendation({}),
                        "orch_second_agree": False,
                        "llm_status": "failed_fallback", "llm_error": str(e)}

    if "E" in RUNGS_TO_RUN and "E" not in out and "D" in out:
        try:
            out["E"] = run_rung_E(pid, evidence_pack, nb06_plan, out["D"])
        except LLMError as e:
            out["E"] = {"orchestrator": out["D"]["orchestrator"],
                        "second_opinion": out["D"]["second_opinion"],
                        "orch_second_agree": out["D"].get("orch_second_agree", False),
                        "redteam_findings": [], "redteam_pass": None,
                        "revisions_made": False, "revision_attempted": False,
                        "defense": "", "revision_reason": "",
                        "llm_status": "failed_fallback", "llm_error": str(e)}

    if "F" in RUNGS_TO_RUN and "F" not in out and "E" in out:
        out["F"] = run_rung_F(pid, evidence_pack, out["E"])

    out["provenance"] = {
        "notebook": NOTEBOOK,
        "llm_model": LLM_MODEL,
        "prompt_version": PROMPT_VERSION,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    return out


print("✓ rung pipelines defined (v3)")

✓ rung pipelines defined (v3)


In [9]:
# ============================================================
# MAIN LOOP — one file per patient, all 6 rungs + activation log
# ============================================================
per_patient_dir = NB_OUTPUT / "per_patient"
per_patient_dir.mkdir(parents=True, exist_ok=True)

run_start = time.time()
completed, skipped_cache, failed, pending = 0, 0, [], []
activation_log = []

print(f"PROMPT_VERSION={PROMPT_VERSION} | model={LLM_MODEL}")
print(f"Rungs: {RUNGS_TO_RUN}")
print(f"Patients: {len(patient_set)} | Expected calls: ~{len(patient_set) * 5}")
print(f"Keys: {len(AVAILABLE_KEYS)}")
print("-" * 70)

for i, pid in enumerate(sorted(patient_set)):
    if MAX_PATIENTS and completed >= MAX_PATIENTS:
        pending.append(pid)
        continue

    out_path = per_patient_dir / f"{pid}.json"

    existing = None
    if out_path.exists():
        try:
            with open(out_path, "r", encoding="utf-8") as f:
                existing = json.load(f)
            if all(r in existing for r in RUNGS_TO_RUN):
                skipped_cache += 1
                completed += 1
                continue
        except Exception:
            existing = None

    if time.time() - run_start > TIME_BUDGET_S:
        pending.append(pid)
        continue

    try:
        result = process_patient_all_rungs(pid, existing=existing)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, default=str)

        # Activation log
        active_agents = []
        if "A" in result: active_agents.append("A")
        if "B" in result: active_agents.append("B")
        if "C" in result: active_agents.append("C_second_opinion")
        if "D" in result: active_agents.append("D_counter")
        if "E" in result: active_agents.append("E_redteam")
        if result.get("E", {}).get("revisions_made"):
            active_agents.append("E_revision")
        if "F" in result: active_agents.append("F_safety_gate")

        activation_log.append({
            "patient_id": pid,
            "rungs_run": [r for r in RUNGS_TO_RUN if r in result],
            "active_agents": active_agents,
            "n_active_agents": len(active_agents),
            "n_llm_calls": sum(1 for a in active_agents if a not in ("F_safety_gate",)),
            "safety_decision": result.get("F", {}).get("safety_gate_decision"),
        })

        completed += 1
        if (i + 1) % 5 == 0 or i == 0:
            calls = llm.calls_this_run if llm else 0
            hits = llm.cache_hits if llm else 0
            rot = llm.rotations if llm else 0
            exh = llm.exhaustion_events if llm else 0
            print(f"[{i+1}/{len(patient_set)}] {pid} OK | "
                  f"calls={calls} cache={hits} rot={rot} exh={exh} "
                  f"elapsed={time.time()-run_start:.0f}s")
    except Exception as e:
        failed.append({"patient_id": pid, "error": str(e)})
        print(f"[{i+1}] {pid} FAILED: {e}")

    if LLM_MODE == "api" and llm is not None:
        time.sleep(SLEEP_BETWEEN_CALLS)

# Persist activation log
if activation_log:
    pd.DataFrame(activation_log).to_csv(NB_OUTPUT / "nb09_activation_log.csv", index=False)

summary = {
    "completed": completed,
    "skipped_from_cache": skipped_cache,
    "failed": failed,
    "pending": pending,
    "llm_calls_this_run": llm.calls_this_run if llm else 0,
    "cache_hits": llm.cache_hits if llm else 0,
    "json_repairs": llm.json_repairs if llm else 0,
    "key_rotations": llm.rotations if llm else 0,
    "exhaustion_events": llm.exhaustion_events if llm else 0,
    "key_calls": llm.key_calls if llm else {},
    "exhausted_keys": [n for n, e in (llm.exhausted.items() if llm else []) if e],
    "tokens_in":  llm.tokens_in  if llm else 0,
    "tokens_out": llm.tokens_out if llm else 0,
    "elapsed_s": round(time.time() - run_start, 1),
    "llm_model": LLM_MODEL,
    "prompt_version": PROMPT_VERSION,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
with open(NB_OUTPUT / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 70)
print(json.dumps(summary, indent=2))
print("=" * 70)

PROMPT_VERSION=09_v4 | model=gemini-3.5-flash-lite
Rungs: ['A', 'B', 'C', 'D', 'E', 'F']
Patients: 98 | Expected calls: ~490
Keys: 2
----------------------------------------------------------------------
[1/98] TCGA-05-4244 OK | calls=7 cache=0 rot=0 exh=0 elapsed=7s
    ↻ rotated to GEMINI_API_KEY_2
    JSON repaired (attempt 1)
[5/98] TCGA-05-4382 OK | calls=32 cache=0 rot=1 exh=0 elapsed=41s
    ↻ rotated to GEMINI_API_KEY
    ↻ rotated to GEMINI_API_KEY_2
    ↻ rotated to GEMINI_API_KEY
    ⛔ GEMINI_API_KEY exhausted after 3 429s
    ↻ rotated to GEMINI_API_KEY_2
    ⛔ GEMINI_API_KEY_2 exhausted after 3 429s
    all keys exhausted — sleeping 60s
    🔄 reset exhaustion flags for all keys
    ↻ rotated to GEMINI_API_KEY
[10/98] TCGA-05-4397 OK | calls=65 cache=0 rot=6 exh=2 elapsed=147s
    ↻ rotated to GEMINI_API_KEY_2
    ↻ rotated to GEMINI_API_KEY
    ↻ rotated to GEMINI_API_KEY_2
    ↻ rotated to GEMINI_API_KEY
[15/98] TCGA-05-4410 OK | calls=98 cache=0 rot=10 exh=2 elapsed=190s

In [10]:
# ============================================================
# MASTER MERGE — collect per-patient, per-rung results  (v3)
# ============================================================
all_patients = {}
for p in sorted(per_patient_dir.glob("*.json")):
    with open(p, "r", encoding="utf-8") as f:
        all_patients[p.stem] = json.load(f)

print(f"loaded {len(all_patients)} patients")


# ------------------------------------------------------------
# Canonicalize treatment names for honest diversity counts
# ------------------------------------------------------------
_DRUG_ALIASES = {
    "carboplatin plus pemetrexed": "carboplatin+pemetrexed",
    "carboplatin + pemetrexed":     "carboplatin+pemetrexed",
    "carboplatin-pemetrexed":       "carboplatin+pemetrexed",
    "cisplatin plus pemetrexed":    "cisplatin+pemetrexed",
    "cisplatin + pemetrexed":       "cisplatin+pemetrexed",
    "cisplatin-pemetrexed":         "cisplatin+pemetrexed",
    "platinum-based doublet chemotherapy": "platinum_doublet",
    "platinum doublet":                     "platinum_doublet",
    "adjuvant platinum-based chemotherapy": "platinum_doublet",
    "pembrolizumab monotherapy":            "pembrolizumab",
    "pembrolizumab plus carboplatin and pemetrexed": "pembro+carbo+pem",
    "pembrolizumab plus platinum-based chemotherapy":"pembro+platinum",
    "carboplatin + pemetrexed + pembrolizumab":      "pembro+carbo+pem",
    "carboplatin+pemetrexed+pembrolizumab":          "pembro+carbo+pem",
}

def canonical_tx(s):
    t = str(s).lower().strip()
    t = re.sub(r"\(.*?\)", "", t)
    t = re.sub(r"\s+", " ", t)
    return _DRUG_ALIASES.get(t, t)


master_rows = []

for pid, r in all_patients.items():
    # Rung A
    if "A" in r:
        rec = r["A"].get("recommendation", {})
        master_rows.append({
            "patient_id": pid, "rung": "A",
            "primary_treatment": str(rec.get("recommended_treatment", ""))[:120],
            "primary_confidence": float(rec.get("confidence", 0.0) or 0.0),
            "primary_escalation": bool(rec.get("escalation_flag", False)),
            "counter_acknowledged_n": 0,
            "rationale_cites_counter": None,
            "second_opinion_treatment": "",
            "second_opinion_confidence": None,
            "orch_second_agree": None,
            "redteam_pass": None,
            "n_redteam_high_critical": None,
            "revision_attempted": None,
            "revisions_made": None,
            "defense_len": None,
            "safety_gate_decision": None,
            "llm_status": r["A"].get("llm_status", "unknown"),
        })
    # Rung B
    if "B" in r:
        rec = r["B"].get("recommendation", {})
        master_rows.append({
            "patient_id": pid, "rung": "B",
            "primary_treatment": str(rec.get("recommended_treatment", ""))[:120],
            "primary_confidence": float(rec.get("confidence", 0.0) or 0.0),
            "primary_escalation": bool(rec.get("escalation_flag", False)),
            "counter_acknowledged_n": 0,
            "rationale_cites_counter": None,
            "second_opinion_treatment": "",
            "second_opinion_confidence": None,
            "orch_second_agree": None,
            "redteam_pass": None,
            "n_redteam_high_critical": None,
            "revision_attempted": None,
            "revisions_made": None,
            "defense_len": None,
            "safety_gate_decision": None,
            "llm_status": r["B"].get("llm_status", "unknown"),
        })
    # Rungs C, D
    for rung in ("C", "D"):
        if rung in r:
            orch = r[rung].get("orchestrator", {})
            so = r[rung].get("second_opinion", {})
            cack = orch.get("counter_evidence_acknowledged", []) or []
            soc_counter = so.get("counter_evidence_considered", []) or []
            master_rows.append({
                "patient_id": pid, "rung": rung,
                "so_counter_considered_n": len(soc_counter),
                "primary_treatment": str(orch.get("recommended_treatment", ""))[:120],
                "primary_confidence": float(orch.get("confidence", 0.0) or 0.0),
                "primary_escalation": bool(orch.get("escalation_flag", False)),
                "counter_acknowledged_n": len(cack),
                "rationale_cites_counter": orch.get("rationale_cites_counter"),
                "second_opinion_treatment": str(so.get("recommended_treatment", ""))[:120],
                "second_opinion_confidence": float(so.get("confidence", 0.0) or 0.0),
                "orch_second_agree": bool(r[rung].get("orch_second_agree", False)),
                "redteam_pass": None,
                "n_redteam_high_critical": None,
                "revision_attempted": None,
                "revisions_made": None,
                "defense_len": None,
                "safety_gate_decision": None,
                "llm_status": r[rung].get("llm_status", "unknown"),
            })
    # Rungs E, F
    for rung in ("E", "F"):
        if rung in r:
            orch = r[rung].get("orchestrator", {})
            so = r[rung].get("second_opinion", {})
            findings = r[rung].get("redteam_findings", []) or []
            n_high_crit = sum(1 for f in findings if f.get("severity") in ("high", "critical"))
            cack = orch.get("counter_evidence_acknowledged", []) or []
            master_rows.append({
                "patient_id": pid, "rung": rung,
                "primary_treatment": str(orch.get("recommended_treatment", ""))[:120],
                "primary_confidence": float(orch.get("confidence", 0.0) or 0.0),
                "primary_escalation": bool(orch.get("escalation_flag", False)),
                "counter_acknowledged_n": len(cack),
                "rationale_cites_counter": orch.get("rationale_cites_counter"),
                "second_opinion_treatment": str(so.get("recommended_treatment", ""))[:120],
                "second_opinion_confidence": float(so.get("confidence", 0.0) or 0.0),
                "orch_second_agree": bool(r[rung].get("orch_second_agree", False)),
                "redteam_pass": r[rung].get("redteam_pass"),
                "n_redteam_high_critical": n_high_crit,
                "revision_attempted": r[rung].get("revision_attempted"),
                "revisions_made": r[rung].get("revisions_made"),
                "defense_len": len(str(r[rung].get("defense", "") or "")),
                "safety_gate_decision": r[rung].get("safety_gate_decision"),
                "llm_status": r[rung].get("llm_status", "unknown"),
            })

master_df = pd.DataFrame(master_rows).sort_values(["patient_id", "rung"]).reset_index(drop=True)
master_df["tx_canonical"] = master_df["primary_treatment"].map(canonical_tx)
master_df.to_csv(NB_OUTPUT / "nb09_master_results.csv", index=False)

print(f"Master results: {len(master_df)} rows | rungs: {sorted(master_df['rung'].unique())}")
display(master_df.head(15))


# ------------------------------------------------------------
# Ablation table (with canonicalized diversity)
# ------------------------------------------------------------
ablation_rows = []
for rung in sorted(master_df["rung"].unique()):
    sub = master_df[master_df["rung"] == rung]
    ablation_rows.append({
        "system": rung,
        "n": int(len(sub)),
        "mean_primary_confidence": round(float(sub["primary_confidence"].mean()), 4),
        "primary_escalation_rate": round(float(sub["primary_escalation"].mean()), 4),
        "agreement_rate": (
            round(float(sub["orch_second_agree"].dropna().mean()), 4)
            if sub["orch_second_agree"].notna().any() else None),
        "counter_ack_rate": (
            round(float((sub["counter_acknowledged_n"] > 0).mean()), 4)
            if sub["counter_acknowledged_n"].notna().any() else None),
        "redteam_pass_rate": (
            round(float(sub["redteam_pass"].dropna().astype(bool).mean()), 4)
            if sub["redteam_pass"].notna().any() else None),
        "revisions_made_rate": (
            round(float(sub["revisions_made"].dropna().astype(bool).mean()), 4)
            if sub["revisions_made"].notna().any() else None),
        "referral_rate": (
            round(float((sub["safety_gate_decision"] == "ESCALATE").mean()), 4)
            if sub["safety_gate_decision"].notna().any() else None),
        "unique_treatments_canonical": int(sub["tx_canonical"].nunique()),
        "unique_treatments_raw":       int(sub["primary_treatment"].nunique()),
    })
ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(NB_OUTPUT / "nb09_ablation_table.csv", index=False)

print("\n=== ABLATION TABLE (canonical) ===")
display(ablation_df)


# ------------------------------------------------------------
# QC
# ------------------------------------------------------------
qc = {
    "notebook": NOTEBOOK,
    "n_patients": N_PATIENTS,
    "n_completed": len(all_patients),
    "rungs": RUNGS_TO_RUN,
    "ablation_table": ablation_df.to_dict(orient="records"),
    "llm_model": LLM_MODEL,
    "prompt_version": PROMPT_VERSION,
    "calls_this_run": llm.calls_this_run if llm else 0,
    "cache_hits": llm.cache_hits if llm else 0,
    "key_rotations": llm.rotations if llm else 0,
    "exhaustion_events": llm.exhaustion_events if llm else 0,
    "tokens_in":  llm.tokens_in  if llm else 0,
    "tokens_out": llm.tokens_out if llm else 0,
    "key_calls": llm.key_calls if llm else {},
    "limitations": [
        "All rungs use the same model; ablation deltas reflect only component additions.",
        "Rung C reuses rung B's orchestrator (no duplicate API call).",
        "Rung E reuses rung D's outputs; rung F is deterministic.",
        "No expert labels — accuracy cannot be computed.",
        "Second opinion is blind to primary (enforced by function signature).",
        "v2: C and D use IDENTICAL system prompts; only COUNTER_HITS block differs.",
        "v3: Counter instruction moved to user prompt; revision agent must defend or change.",
        "v3: treatment-name canonicalization applied for diversity counts.",
    ],
    "patient_leakage_note": "all artifacts keyed by patient_id",
}
with open(NB_OUTPUT / "nb09_qc_report.json", "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2, default=str)

print("\n✓ master merge complete (v3)")

loaded 98 patients
Master results: 588 rows | rungs: ['A', 'B', 'C', 'D', 'E', 'F']


,patient_id,rung,primary_treatment,primary_confidence,primary_escalation,counter_acknowledged_n,rationale_cites_counter,second_opinion_treatment,second_opinion_confidence,orch_second_agree,redteam_pass,n_redteam_high_critical,revision_attempted,revisions_made,defense_len,safety_gate_decision,llm_status,so_counter_considered_n,tx_canonical
0,TCGA-05-4244,A,"Carboplatin, pemetrexed, and pembrolizumab",0.80,False,0,None,,NaN,None,None,NaN,None,None,NaN,None,api_success,NaN,"carboplatin, pemetrexed, and pembrolizumab"
1,TCGA-05-4244,B,Sotorasib,0.65,False,0,None,,NaN,None,None,NaN,None,None,NaN,None,api_success,NaN,sotorasib
2,TCGA-05-4244,C,Sotorasib,0.65,False,0,False,Carboplatin+Pemetrexed (platinum doublet),0.85,False,None,NaN,None,None,NaN,None,api_success,1.0,sotorasib
3,TCGA-05-4244,D,Carboplatin+Pemetrexed,0.80,False,1,False,Carboplatin+Pemetrexed (platinum doublet),0.85,True,None,NaN,None,None,NaN,None,api_success,0.0,carboplatin+pemetrexed
4,TCGA-05-4244,E,ESCALATE_TO_MDT,0.90,True,0,False,Carboplatin+Pemetrexed (platinum doublet),0.85,True,False,1.0,True,True,0.0,None,api_success,NaN,escalate_to_mdt
5,TCGA-05-4244,F,ESCALATE_TO_MDT,0.90,True,0,False,Carboplatin+Pemetrexed (platinum doublet),0.85,True,False,1.0,None,None,0.0,ACCEPT,api_success,NaN,escalate_to_mdt
6,TCGA-05-4245,A,Carboplatin + Pemetrexed,0.60,False,0,None,,NaN,None,None,NaN,None,None,NaN,None,api_success,NaN,carboplatin+pemetrexed
7,TCGA-05-4245,B,Carboplatin+Pemetrexed,0.85,False,0,None,,NaN,None,None,NaN,None,None,NaN,None,api_success,NaN,carboplatin+pemetrexed
8,TCGA-05-4245,C,Carboplatin+Pemetrexed,0.85,False,0,False,Carboplatin+Pemetrexed (platinum doublet),0.80,True,None,NaN,None,None,NaN,None,api_success,0.0,carboplatin+pemetrexed
9,TCGA-05-4245,D,Carboplatin+Pemetrexed,0.85,False,1,False,Carboplatin+Pemetrexed (platinum doublet),0.80,True,None,NaN,None,None,NaN,None,api_success,2.0,carboplatin+pemetrexed



=== ABLATION TABLE (canonical) ===


,system,n,mean_primary_confidence,primary_escalation_rate,agreement_rate,counter_ack_rate,redteam_pass_rate,revisions_made_rate,referral_rate,unique_treatments_canonical,unique_treatments_raw
0,A,98,0.7148,0.0000,NaN,0.0000,NaN,NaN,NaN,22,34
1,B,98,0.7995,0.0102,NaN,0.0000,NaN,NaN,NaN,11,11
2,C,98,0.7995,0.0102,0.6122,0.0000,NaN,NaN,NaN,11,11
3,D,98,0.8170,0.0306,0.5510,0.5306,NaN,NaN,NaN,9,9
4,E,98,0.8471,0.4898,0.5510,0.1837,0.3265,0.551,NaN,6,6
5,F,98,0.8471,0.4898,0.5510,0.1837,0.3265,NaN,0.5,6,6



✓ master merge complete (v3)


In [11]:
# ============================================================
# NB09 DIAGNOSTIC — read-only, no LLM calls  (v3)
# ============================================================
import json, re
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
import numpy as np

per_patient_dir = NB_OUTPUT / "per_patient"
files = sorted(per_patient_dir.glob("*.json"))
all_patients = {p.stem: json.load(open(p)) for p in files}
print(f"loaded {len(all_patients)} patient files\n")


# --- Q1: counter quality ---
print("=" * 70); print("Q1. COUNTER-EVIDENCE QUALITY"); print("=" * 70)
NEG_TERMS = ["resistan","contraindicat","fail","refractory","adverse","toxicity",
             "no benefit","did not improve","worse","negative","not recommended",
             "should not","avoid","discontinu","progression","relapse","non-responder"]
POS_TERMS = ["effective","improved","benefit","response","sensitiz","approved",
             "recommended","first-line","standard of care"]
rows = []
for pid in list(all_patients.keys())[:30]:
    pack = load_evidence_pack(pid)
    for h in pack.get("counter_hits", []):
        text = (str(h.get("title","")) + " " + str(h.get("abstract",""))).lower()
        n_neg = sum(text.count(t) for t in NEG_TERMS)
        n_pos = sum(text.count(t) for t in POS_TERMS)
        rows.append({"patient_id": pid, "source": h.get("source"),
                     "source_id": h.get("source_id"),
                     "title": str(h.get("title",""))[:80],
                     "net_signal": n_neg - n_pos})
if rows:
    ce = pd.DataFrame(rows)
    print(f"counter hits sampled: {len(ce)} across 30 patients")
    print(f"  adversarial: {(ce['net_signal']>0).mean():.1%}")
    print(f"  neutral:     {(ce['net_signal']==0).mean():.1%}")
    print(f"  supportive:  {(ce['net_signal']<0).mean():.1%}  <-- JUNK if high")
    print("\nBy source:")
    print(ce.groupby("source")["net_signal"].agg(["mean","count"]).round(2).to_string())
else:
    ce = pd.DataFrame(columns=["net_signal"])


# --- Q2: rung A ---
print("\n" + "=" * 70); print("Q2. RUNG A ESCALATION"); print("=" * 70)
A_esc, A_rec, A_conf = [], [], []
for pid, r in all_patients.items():
    if "A" in r:
        rec = r["A"].get("recommendation", {})
        A_esc.append(bool(rec.get("escalation_flag", False)))
        A_rec.append(str(rec.get("recommended_treatment", "")))
        A_conf.append(float(rec.get("confidence", 0.0) or 0.0))
if A_esc:
    print(f"Rung A escalation rate: {np.mean(A_esc):.1%} ({sum(A_esc)}/{len(A_esc)})")
    if any(A_esc):
        print(f"conf on escalated: {np.mean([c for c,e in zip(A_conf,A_esc) if e]):.2f}")
    if any(not e for e in A_esc):
        print(f"conf on non-esc:   {np.mean([c for c,e in zip(A_conf,A_esc) if not e]):.2f}")
    print("Non-escalated treatments:", Counter([t for t,e in zip(A_rec,A_esc) if not e]).most_common(10))


# --- Q3: C -> D ---
print("\n" + "=" * 70); print("Q3. C -> D FLIP ANALYSIS"); print("=" * 70)
def _n(s): return re.sub(r"\s+"," ", re.sub(r"\(.*?\)","", str(s).lower())).strip()
flips_esc, flips_rec, counter_ack = 0, 0, 0
examples = []
for pid, r in all_patients.items():
    if "C" not in r or "D" not in r: continue
    c, d = r["C"].get("orchestrator",{}), r["D"].get("orchestrator",{})
    c_esc, d_esc = bool(c.get("escalation_flag")), bool(d.get("escalation_flag"))
    if c_esc != d_esc:
        flips_esc += 1
        if len(examples) < 5:
            examples.append((pid, _n(c.get("recommended_treatment","")), _n(d.get("recommended_treatment",""))))
    elif _n(c.get("recommended_treatment","")) != _n(d.get("recommended_treatment","")):
        flips_rec += 1
    if (d.get("counter_evidence_acknowledged") or []):
        counter_ack += 1
n = sum(1 for r in all_patients.values() if "C" in r and "D" in r)
if n:
    print(f"n = {n}")
    print(f"  escalation flips C->D:       {flips_esc} ({flips_esc/n:.1%})")
    print(f"  treatment flips (same esc):  {flips_rec} ({flips_rec/n:.1%})")
    print(f"  D acknowledged counter:      {counter_ack} ({counter_ack/n:.1%})")
    for e in examples:
        print(f"  {e[0]}: C='{e[1][:35]}' -> D='{e[2][:35]}'")

# --- NEW: does the second-opinion see counter in rung D? ---
print("\n" + "=" * 70); print("Q3b. SECOND OPINION × COUNTER AWARENESS (rung D)"); print("=" * 70)
so_counter_hit = 0
so_counter_mention = 0
so_total = 0
for pid, r in all_patients.items():
    if "D" not in r: continue
    so = r["D"].get("second_opinion", {})
    so_total += 1
    cits = so.get("supporting_citations", []) or []
    if any("counter" in str(c).lower() for c in cits):
        so_counter_hit += 1
    if "counter" in str(so.get("rationale","")).lower():
        so_counter_mention += 1
so_counter_considered = 0
for pid, r in all_patients.items():
    if "D" not in r: continue
    so = r["D"].get("second_opinion", {})
    if so.get("counter_evidence_considered"):
        so_counter_considered += 1
if so_total:
    print(f"cited counter in supporting_citations: {so_counter_hit}/{so_total} ({so_counter_hit/so_total:.1%})")
    print(f"mentioned 'counter' in rationale:      {so_counter_mention}/{so_total} ({so_counter_mention/so_total:.1%})")
    print(f"counter_evidence_considered non-empty: {so_counter_considered}/{so_total} ({so_counter_considered/so_total:.1%})")
    print("  ^ if all ~0, second-opinion is not receiving or using the counter block")


# --- Q4: rung E inertness ---
print("\n" + "=" * 70); print("Q4. RUNG E INERTNESS"); print("=" * 70)
sev_counter, survived, rev_made, rev_attempt = Counter(), Counter(), 0, 0
defense_lens = []
for pid, r in all_patients.items():
    if "E" not in r: continue
    if r["E"].get("revisions_made"): rev_made += 1
    if r["E"].get("revision_attempted"): rev_attempt += 1
    dl = len(str(r["E"].get("defense", "") or ""))
    if dl: defense_lens.append(dl)
    for f in r["E"].get("redteam_findings", []) or []:
        sev_counter[f.get("severity","?")] += 1
        if f.get("survives"):
            survived[f.get("severity","?")] += 1
print("severity all:", dict(sev_counter))
print("severity survived:", dict(survived))
print(f"cases with revision_attempted=True: {rev_attempt}")
print(f"cases with revisions_made=True:     {rev_made}")
if defense_lens:
    print(f"defense length (chars): mean={np.mean(defense_lens):.0f} min={min(defense_lens)} max={max(defense_lens)}")

print()
print("--- FULL DEFENSE TEXTS (to check for caching bug) ---")
for pid, r in all_patients.items():
    if "E" in r and r["E"].get("defense"):
        d = r["E"]["defense"]
        print(f"\n[{pid}] len={len(d)}")
        print(d)
        print("-" * 60)

diff_E_D = 0
for pid, r in all_patients.items():
    if "D" not in r or "E" not in r: continue
    if _n(r["D"].get("orchestrator",{}).get("recommended_treatment","")) != \
       _n(r["E"].get("orchestrator",{}).get("recommended_treatment","")):
        diff_E_D += 1
print(f"cases where E's recommendation differs from D's: {diff_E_D}")


# --- Q5: rung F ---
print("\n" + "=" * 70); print("Q5. RUNG F SAFETY GATE"); print("=" * 70)
reason_counter, by_decision = Counter(), Counter()
for pid, r in all_patients.items():
    if "F" not in r: continue
    by_decision[r["F"].get("safety_gate_decision","?")] += 1
    for reason in r["F"].get("safety_gate_reasons", []) or []:
        reason_counter[reason] += 1
n_f = sum(by_decision.values())
for d, c in by_decision.items(): print(f"  {d}: {c} ({c/n_f:.1%})")
print("Reasons:")
for reason, count in reason_counter.most_common():
    print(f"  {count:3d} ({count/n_f:5.1%})  {reason}")


# --- SUMMARY ---
print("\n" + "=" * 70); print("SUMMARY — red flags"); print("=" * 70)
flags = []
if len(ce) and ce["net_signal"].mean() <= 0.5:
    flags.append(f"counter mean net signal = {ce['net_signal'].mean():.2f}")
if A_esc and np.mean(A_esc) > 0.5:
    flags.append(f"rung A escalation {np.mean(A_esc):.1%}")
if n and flips_esc / n > 0.3:
    flags.append(f"{flips_esc/n:.1%} escalation flips C->D")
if n and flips_rec / n > 0.3:
    flags.append(f"{flips_rec/n:.1%} treatment flips C->D")
if so_total and so_counter_hit / so_total < 0.05:
    flags.append(f"second opinion cited counter in only {so_counter_hit/so_total:.1%} of rung-D cases")
if diff_E_D == 0:
    flags.append("rung E identical to D in 100% of cases")
if by_decision.get("ESCALATE",0) / max(n_f,1) > 0.5:
    flags.append(f"rung F referral at {by_decision['ESCALATE']/n_f:.1%}")

if flags:
    for f in flags: print(f"  [!] {f}")
else:
    print("  ✓ no red flags")

loaded 98 patient files

Q1. COUNTER-EVIDENCE QUALITY
counter hits sampled: 150 across 30 patients
  adversarial: 71.3%
  neutral:     28.7%
  supportive:  0.0%  <-- JUNK if high

By source:
                mean  count
source                     
civic           0.00      4
clinicaltrials  1.00     25
guideline       0.00      6
pubmed          0.71    115

Q2. RUNG A ESCALATION
Rung A escalation rate: 0.0% (0/98)
conf on non-esc:   0.71
Non-escalated treatments: [('Pembrolizumab', 20), ('Carboplatin plus pemetrexed', 10), ('Carboplatin + Pemetrexed', 9), ('Platinum-based doublet chemotherapy', 7), ('Pembrolizumab plus carboplatin and pemetrexed', 5), ('Pembrolizumab monotherapy', 5), ('Adjuvant platinum-based doublet chemotherapy', 4), ('Carboplatin-Pemetrexed', 3), ('Carboplatin-pemetrexed', 2), ('Osimertinib', 2)]

Q3. C -> D FLIP ANALYSIS
n = 98
  escalation flips C->D:       2 (2.0%)
  treatment flips (same esc):  8 (8.2%)
  D acknowledged counter:      52 (53.1%)
  TCGA-38-6178: 

In [12]:
import gc
import shutil
from pathlib import Path

outputs = [{"file": str(p.relative_to(NB_OUTPUT)), "size_bytes": p.stat().st_size}
           for p in sorted(NB_OUTPUT.rglob("*")) if p.is_file()]
display(pd.DataFrame(outputs))

zip_stem = OUTPUT_ROOT / "09_artifacts"
zip_path = Path(str(zip_stem) + ".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_stem), "zip", str(NB_OUTPUT))

print("\n" + "=" * 55)
print("VERIONCO NB09 (UNIFIED ABLATION v3) COMPLETE")
print("=" * 55)
print(f"Artifacts:      {NB_OUTPUT}")
print(f"Archive:        {zip_path}")
print(f"Master table:   {NB_OUTPUT / 'nb09_master_results.csv'}")
print(f"Ablation:       {NB_OUTPUT / 'nb09_ablation_table.csv'}")
print(f"Activation log: {NB_OUTPUT / 'nb09_activation_log.csv'}")
print(f"API calls:      {llm.calls_this_run if llm else 0}")
print(f"Cache hits:     {llm.cache_hits if llm else 0}")
print(f"Tokens in:      {llm.tokens_in if llm else 0}")
print(f"Tokens out:     {llm.tokens_out if llm else 0}")
print("=" * 55)

gc.collect()

,file,size_bytes
0,nb09_ablation_table.csv,429
1,nb09_activation_log.csv,13967
2,nb09_master_results.csv,72676
3,nb09_qc_report.json,3327
4,per_patient/TCGA-05-4244.json,11064
...,...,...
98,per_patient/TCGA-44-A4SS.json,11575
99,per_patient/TCGA-44-A4SU.json,9814
100,per_patient/TCGA-49-4487.json,12449
101,per_patient/TCGA-49-4488.json,12491



VERIONCO NB09 (UNIFIED ABLATION v3) COMPLETE
Artifacts:      /kaggle/working/VERIONCO_outputs/09
Archive:        /kaggle/working/VERIONCO_outputs/09_artifacts.zip
Master table:   /kaggle/working/VERIONCO_outputs/09/nb09_master_results.csv
Ablation:       /kaggle/working/VERIONCO_outputs/09/nb09_ablation_table.csv
Activation log: /kaggle/working/VERIONCO_outputs/09/nb09_activation_log.csv
API calls:      654
Cache hits:     0
Tokens in:      641483
Tokens out:     114780


94

In [ ]:
# ============================================================
# EXPORT & DOWNLOAD ARTIFACT BUNDLE
# ============================================================
import shutil
from pathlib import Path
try:
    from IPython.display import FileLink, display
except ImportError:
    def FileLink(p): return p
    def display(*a): pass

export_dir = OUTPUT_ROOT / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
zip_name = f'verionco_outputs_{NB_OUTPUT.name}'
zip_path = export_dir / zip_name

# Package NB_OUTPUT into zip
shutil.make_archive(str(zip_path), 'zip', NB_OUTPUT)
final_zip = export_dir / f'{zip_name}.zip'

print('=' * 60)
print(f'VERIONCO NOTEBOOK {NB_OUTPUT.name} COMPLETE')
print(f'Artifact bundle created: {final_zip}')
print(f'Archive size: {final_zip.stat().st_size / (1024*1024):.2f} MB')
print('=' * 60)

# Display download link (clickable in Kaggle/Jupyter)
display(FileLink(str(final_zip)))
